# Workflow 15 - EfficientNet family, OPTIMISED fine-tuning
## BRISC 2025 - fair same-model comparison with the dataset paper

**Catarina Bota | UCP**

Our uniform two-stage protocol (freeze base, then unfreeze only the top-60 layers at lr 1e-5) under-fits EfficientNet: on the audited partition B0 reached **89.1%** and B2 **93.9%**, far below ResNet50/VGG16 (~98%) and below the BRISC paper's own EfficientNetB0 baseline (**99.20%** on the raw release). The documented cause is BatchNormalization: unfreezing EfficientNet's BN layers on a small dataset destroys the pretrained statistics.

This notebook re-trains the EfficientNet family with the **optimised** recipe from the Keras fine-tuning guide: **unfreeze the whole backbone but keep every BatchNormalization layer frozen**, with a higher fine-tune learning rate (1e-4) and more epochs (up to 30).

> The uniform-protocol results (W8, W14) are **not** overwritten. Optimised outputs go to `efficientnet_optimized/`. This notebook is a pure orchestrator - it does **not** import TensorFlow; each model trains in an isolated subprocess (`train_efficientnet_optimized.py`) so VRAM is fully released between runs.

**How to run:** confirm the kernel is the WSL `/root/brisc/.venv`, then **Run All**. Each (variant x partition) takes ~15-25 min. The default B0+B1+B2 x audited+unaudited = 6 runs (~2 h on an RTX 4060). Trim `ACTIVE_VARIANTS` / `PARTITIONS` below to shorten.

## 1. Configuration

In [1]:
import os
base = '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/efficientnet_optimized'
for v in ['B0', 'B1', 'B2']:
    p = f'{base}/{v}/audited/metrics.json'
    if os.path.exists(p):
        os.remove(p)
        print('removido', p)
    else:
        print('did not exist', p)

removido /mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/efficientnet_optimized/B0/audited/metrics.json
removido /mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/efficientnet_optimized/B1/audited/metrics.json
removido /mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/efficientnet_optimized/B2/audited/metrics.json


In [2]:
import os, sys, json, subprocess
import pandas as pd

PROJECT_DIR = '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui'
WORKER      = f'{PROJECT_DIR}/train_efficientnet_optimized.py'
OUT_ROOT    = f'{PROJECT_DIR}/efficientnet_optimized'
os.makedirs(OUT_ROOT, exist_ok=True)

DATASETS = {
    'audited':   '/root/brisc/data/brisc2025_clean/classification_task',
    'unaudited': '/root/brisc/data_unaudited/classification_task',
}

# B0-B2 are safe on 8 GB. B3 (300px) / B4 (380px) are heavier -- add only if VRAM allows.
ACTIVE_VARIANTS = ['B0', 'B1', 'B2']
PARTITIONS      = ['audited', 'unaudited']

assert os.path.exists(WORKER), f'Worker not found: {WORKER}'
print('Python  :', sys.executable)
print('Variants:', ACTIVE_VARIANTS, '| Partitions:', PARTITIONS)
print('Output  :', OUT_ROOT)

Python  : /root/brisc/.venv/bin/python3
Variants: ['B0', 'B1', 'B2'] | Partitions: ['audited', 'unaudited']
Output  : /mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/efficientnet_optimized


## 2. Train each (variant x partition) in an isolated subprocess

Skips any run whose `metrics.json` already exists, so you can re-run safely or resume.

In [3]:
for variant in ACTIVE_VARIANTS:
    for part in PARTITIONS:
        mpath = f'{OUT_ROOT}/{variant}/{part}/metrics.json'
        if os.path.exists(mpath):
            print(f'  [skip] {variant} / {part} -- already done')
            continue
        print('\n' + '#'*70)
        print(f'# TRAIN  EfficientNet{variant}  /  {part}')
        print('#'*70, flush=True)
        subprocess.run([sys.executable, WORKER, variant, part,
                        DATASETS[part], OUT_ROOT], check=True)


######################################################################
# TRAIN  EfficientNetB0  /  audited
######################################################################


E0000 00:00:1783547229.988755    9723 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783547230.012399    9723 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783547230.188176    9723 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783547230.188238    9723 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783547230.188242    9723 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783547230.188243    9723 computation_placer.cc:177] computation placer already registered. Please check linka

  EfficientNetB0  (audited)  size=224 batch=16  OPTIMISED
Found 3491 validated image filenames belonging to 4 classes.
Found 872 validated image filenames belonging to 4 classes.
Found 678 validated image filenames belonging to 4 classes.
Epoch 1/5


I0000 00:00:1783547245.711820    9808 cuda_dnn.cc:529] Loaded cuDNN version 90300


219/219 - 48s - 221ms/step - accuracy: 0.7496 - loss: 0.6279 - val_accuracy: 0.8589 - val_loss: 0.3847
Epoch 2/5
219/219 - 34s - 157ms/step - accuracy: 0.8356 - loss: 0.4146 - val_accuracy: 0.8956 - val_loss: 0.3083
Epoch 3/5
219/219 - 36s - 163ms/step - accuracy: 0.8510 - loss: 0.3682 - val_accuracy: 0.8933 - val_loss: 0.3031
Epoch 4/5
219/219 - 36s - 166ms/step - accuracy: 0.8742 - loss: 0.3412 - val_accuracy: 0.8968 - val_loss: 0.2545
Epoch 5/5
219/219 - 34s - 156ms/step - accuracy: 0.8714 - loss: 0.3209 - val_accuracy: 0.9117 - val_loss: 0.2571
Restoring model weights from the end of the best epoch: 5.
  unfroze backbone, kept 49 BatchNorm layers frozen
Epoch 1/30


...

  EfficientNetB2  (audited)  size=260 batch=8  OPTIMISED
Found 3491 validated image filenames belonging to 4 classes.
Found 872 validated image filenames belonging to 4 classes.
Found 678 validated image filenames belonging to 4 classes.
Epoch 1/5


I0000 00:00:1783549984.851494   14079 cuda_dnn.cc:529] Loaded cuDNN version 90300


437/437 - 65s - 148ms/step - accuracy: 0.7806 - loss: 0.5652 - val_accuracy: 0.8979 - val_loss: 0.3100
Epoch 2/5
437/437 - 52s - 120ms/step - accuracy: 0.8476 - loss: 0.3938 - val_accuracy: 0.9071 - val_loss: 0.2575
Epoch 3/5
437/437 - 54s - 122ms/step - accuracy: 0.8634 - loss: 0.3525 - val_accuracy: 0.9106 - val_loss: 0.2409
Epoch 4/5
437/437 - 55s - 127ms/step - accuracy: 0.8826 - loss: 0.3181 - val_accuracy: 0.9025 - val_loss: 0.2499
Epoch 5/5
437/437 - 49s - 112ms/step - accuracy: 0.8851 - loss: 0.2986 - val_accuracy: 0.9128 - val_loss: 0.2202
Restoring model weights from the end of the best epoch: 5.
  unfroze backbone, kept 69 BatchNorm layers frozen
Epoch 1/30


E0000 00:00:1783550265.317937   13997 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/efficientnetb2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


437/437 - 92s - 210ms/step - accuracy: 0.9000 - loss: 0.2696 - val_accuracy: 0.9564 - val_loss: 0.1333 - learning_rate: 1.0000e-04
Epoch 2/30
437/437 - 73s - 168ms/step - accuracy: 0.9510 - loss: 0.1360 - val_accuracy: 0.9599 - val_loss: 0.1023 - learning_rate: 1.0000e-04
Epoch 3/30
437/437 - 73s - 166ms/step - accuracy: 0.9671 - loss: 0.0907 - val_accuracy: 0.9587 - val_loss: 0.1427 - learning_rate: 1.0000e-04
Epoch 4/30
437/437 - 73s - 168ms/step - accuracy: 0.9725 - loss: 0.0791 - val_accuracy: 0.9610 - val_loss: 0.1569 - learning_rate: 1.0000e-04
Epoch 5/30
437/437 - 74s - 169ms/step - accuracy: 0.9777 - loss: 0.0649 - val_accuracy: 0.9828 - val_loss: 0.0579 - learning_rate: 1.0000e-04
Epoch 6/30
437/437 - 69s - 159ms/step - accuracy: 0.9891 - loss: 0.0303 - val_accuracy: 0.9656 - val_loss: 0.1103 - learning_rate: 1.0000e-04
Epoch 7/30
437/437 - 68s - 156ms/step - accuracy: 0.9897 - loss: 0.0338 - val_accuracy: 0.9885 - val_loss: 0.0504 - learning_rate: 1.0000e-04
Epoch 8/30
437/43

## 3. Results (optimised protocol)

In [4]:
rows = []
for variant in ACTIVE_VARIANTS:
    for part in PARTITIONS:
        p = f'{OUT_ROOT}/{variant}/{part}/metrics.json'
        if os.path.exists(p):
            rows.append(json.load(open(p)))
opt = pd.DataFrame(rows)
opt.to_csv(f'{OUT_ROOT}/efficientnet_optimized_results.csv', index=False)
cols = [c for c in ['variant','dataset','accuracy','f1_macro','auc_ovr','ci95_lo','ci95_hi','epochs_ft'] if c in opt.columns]
print(opt[cols].to_string(index=False) if len(opt) else 'no results yet')
opt[cols] if len(opt) else None

       variant   dataset  accuracy  f1_macro  auc_ovr  ci95_lo  ci95_hi  epochs_ft
EfficientNetB0   audited     98.53     98.58    99.95    97.49    99.41         20
EfficientNetB0 unaudited     99.20     99.28    99.96    98.60    99.70         25
EfficientNetB1   audited     98.82     98.84    99.96    97.94    99.56         18
EfficientNetB1 unaudited     99.40     99.46    99.99    98.90    99.80         17
EfficientNetB2   audited     98.23     98.29    99.96    97.05    99.12         13
EfficientNetB2 unaudited     98.70     98.85    99.97    98.00    99.30         10


,variant,dataset,accuracy,f1_macro,auc_ovr,ci95_lo,ci95_hi,epochs_ft
0,EfficientNetB0,audited,98.53,98.58,99.95,97.49,99.41,20
1,EfficientNetB0,unaudited,99.20,99.28,99.96,98.60,99.70,25
2,EfficientNetB1,audited,98.82,98.84,99.96,97.94,99.56,18
3,EfficientNetB1,unaudited,99.40,99.46,99.99,98.90,99.80,17
4,EfficientNetB2,audited,98.23,98.29,99.96,97.05,99.12,13
5,EfficientNetB2,unaudited,98.70,98.85,99.97,98.00,99.30,10


## 4. Optimised vs uniform protocol vs the BRISC paper

Uniform-protocol numbers come from the leakage suite (`W8`=B2, `W14`=B0). The BRISC paper (Fateh et al. 2026) reports EfficientNetB0 = 99.20% on the **raw/unaudited** release.

In [5]:
def uni(path):
    return json.load(open(path))['accuracy'] if os.path.exists(path) else None
uniform = {
    ('B0','audited'):   uni(f'{PROJECT_DIR}/leakage_backbones/W14_EfficientNetB0_ImageNet/audited/metrics.json'),
    ('B0','unaudited'): uni(f'{PROJECT_DIR}/leakage_backbones/W14_EfficientNetB0_ImageNet/unaudited/metrics.json'),
    ('B2','audited'):   uni(f'{PROJECT_DIR}/leakage_backbones/W8_EfficientNetB2_ImageNet/audited/metrics.json'),
    ('B2','unaudited'): uni(f'{PROJECT_DIR}/leakage_backbones/W8_EfficientNetB2_ImageNet/unaudited/metrics.json'),
}
comp = []
for r in rows:
    v = r['variant'].replace('EfficientNet',''); ds = r['dataset']
    comp.append({'variant': v, 'partition': ds,
                 'uniform_acc': uniform.get((v,ds)),
                 'optimised_acc': r['accuracy'],
                 'gain_pp': (round(r['accuracy']-uniform[(v,ds)],2) if uniform.get((v,ds)) else None)})
cmp = pd.DataFrame(comp)
cmp.to_csv(f'{OUT_ROOT}/optimised_vs_uniform.csv', index=False)
print(cmp.to_string(index=False) if len(cmp) else 'run section 2 first')
print('\nReference: BRISC paper (Fateh et al. 2026) EfficientNetB0 = 99.20% on the raw release.')
cmp if len(cmp) else None

variant partition  uniform_acc  optimised_acc  gain_pp
     B0   audited        89.09          98.53     9.44
     B0 unaudited        92.20          99.20     7.00
     B1   audited          NaN          98.82      NaN
     B1 unaudited          NaN          99.40      NaN
     B2   audited        93.95          98.23     4.28
     B2 unaudited        95.30          98.70     3.40

Reference: BRISC paper (Fateh et al. 2026) EfficientNetB0 = 99.20% on the raw release.


,variant,partition,uniform_acc,optimised_acc,gain_pp
0,B0,audited,89.09,98.53,9.44
1,B0,unaudited,92.20,99.20,7.00
2,B1,audited,NaN,98.82,NaN
3,B1,unaudited,NaN,99.40,NaN
4,B2,audited,93.95,98.23,4.28
5,B2,unaudited,95.30,98.70,3.40


## Notes

- **Interpretation.** If the optimised B0/B1 on the *unaudited* partition approaches the paper's 99.20%, we have replicated their baseline with our protocol; the drop to the *audited* partition then isolates the leakage effect for the exact model the authors highlight. If a gap remains even on unaudited, it reflects further protocol differences (image size, augmentation, class weighting) that the paper did not fully specify.
- **Keep both.** Report the uniform-protocol numbers for the fair cross-architecture comparison (all backbones treated identically) AND the optimised numbers for the same-model comparison with the literature. State the protocol explicitly for each.
- **VRAM.** Close other GPU consumers before running. If B2 OOMs, lower its batch in `train_efficientnet_optimized.py` (`VARIANT['B2']['batch']`).

In [6]:
!cd "/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui" && python w11_gradcam.py W15_EffNetB1_optimised /root/brisc/data/brisc2025_clean/classification_task "/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/w11_hybrid"

/bin/bash: line 1: python: command not found


In [7]:
%run "/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/build_paper_figures.py"

saved fig_workflows


/root/brisc/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


saved fig_confusion
saved fig_gradcam
done -> /mnt/c/Users/cbot/Desktop/BRISC pos graduação/latex_artigo/figures


In [8]:
!cd "/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui" && python export_models.py

/bin/bash: line 1: python: command not found


In [9]:
%cd "/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui"
import sys
!"{sys.executable}" app_multi.py

!pip install PyQt6 

/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui
Traceback (most recent call last):
  File "/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/app_multi.py", line 16, in <module>
    from PyQt6.QtWidgets import (
ModuleNotFoundError: No module named 'PyQt6'
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.14/README.venv for more

In [10]:
import sys
print('PYTHON :', sys.executable)
print('VERSION:', sys.version.split()[0])
for m in ['tensorflow','sklearn','lightgbm','numpy','PIL','PyQt6']:
    try: __import__(m); print('  OK      ', m)
    except Exception: print('  MISSING ', m)

PYTHON : /root/brisc/.venv/bin/python3
VERSION: 3.12.13


2026-07-09 00:56:07.532671: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-09 00:56:07.717260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783554967.799441    9683 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783554967.825323    9683 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783554967.990282    9683 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

  OK       tensorflow
  OK       sklearn
  OK       lightgbm
  OK       numpy
  OK       PIL
  MISSING  PyQt6


In [11]:
import sys
!"{sys.executable}" -m pip install PyQt6

/root/brisc/.venv/bin/python3: No module named pip


In [12]:
import subprocess, sys
subprocess.run([sys.executable,
    '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/build_training_history.py'],
    check=True)

saved fig_training_history
saved fig_training_loss
done -> figures in figuras_artigo/ and latex_artigo/figures/


CompletedProcess(args=['/root/brisc/.venv/bin/python3', '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/build_training_history.py'], returncode=0)